# DROUGHTS

## INTRODUCCIÓN Y OBJETIVOS

Objetivo del estudio

El objetivo de este análisis es estudiar la relación entre los episodios de calor extremo y los periodos de déficit de precipitaciones con la disponibilidad de recursos hídricos en Cataluña.

Para ello, se utilizarán datos meteorológicos de Meteocat (XEMA) y datos hidrológicos de la Agència Catalana de l'Aigua (ACA), incluyendo información sobre las reservas de los embalses y los estados oficiales de sequía.

Se pretende:
- Identificar episodios de temperaturas extremas y déficit de precipitaciones.
- Analizar la evolución temporal de las reservas de agua.
- Estudiar la relación entre las condiciones meteorológicas y la disponibilidad de agua.
- Comparar los indicadores meteorológicos e hidrológicos con los estados oficiales de sequía.
- Identificar posibles retardos temporales entre la falta de precipitaciones y sus efectos sobre los recursos hídricos.

## IMPORTACIÓN LIBRERÍAS

In [1]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid")


## CARGAR LOS DATOS DE LA XEMA

In [2]:
# XEMA - Meteorological daily data

path_xema = r"C:\Users\nmayo\OneDrive\Escritorio\Máster - UB\Análisis de datos masivos\Datasets - Agua\Dades_meteorològiques_diàries_de_la_XEMA_20261116.csv"
df_xema = pd.read_csv(path_xema, low_memory=False)

print("Dimensions:", df_xema.shape)

display(df_xema.head())

Dimensions: (33475032, 10)


,ID,CODI_ESTACIO,NOM_ESTACIO,DATA_LECTURA,CODI_VARIABLE,NOM_VARIABLE,VALOR,UNITAT,HORA _TU,Estat
0,VK1000098801,VK,Raimat,01/09/1988,1.00,Temperatura mitjana diària,"23,3",°C,NaN,Representatiu
1,VK1002098801,VK,Raimat,01/09/1988,1.00,Temperatura mínima diària + hora,"18,3",°C,0:00,Representatiu
2,VK1100098801,VK,Raimat,01/09/1988,1.10,Humitat relativa mitjana diària,58,%,NaN,Representatiu
3,VK1300098801,VK,Raimat,01/09/1988,1.30,Precipitació acumulada diària,"0,6",mm,NaN,Representatiu
4,VK1301098801,VK,Raimat,01/09/1988,1.30,Precipitació acumulada diària (8-8 h),"0,6",mm,NaN,Representatiu


## CARGA DATOS DE LOS EMBALSES DE LA ACA

In [3]:
# ACA - Water quantity in reservoirs

url_aca = "https://analisi.transparenciacatalunya.cat/api/v3/views/gn9e-3qhr/query.json"

def download_socrata(url, limit=1000000):

    frames = []
    offset = 0

    while True:

        params = {
            "$limit": limit,
            "$offset": offset
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        batch = response.json()

        if len(batch) == 0:
            break

        frames.append(pd.DataFrame(batch))
        offset += len(batch)

        print(f"Downloaded {offset} records")

        if len(batch) < limit:
            break

    return pd.concat(frames, ignore_index=True)


df_aca = download_socrata(url_aca)

print("ACA reservoirs loaded successfully")
print("Dimensions:", df_aca.shape)

display(df_aca.head())

Downloaded 88354 records
ACA reservoirs loaded successfully
Dimensions: (88354, 9)


,:id,:version,:created_at,:updated_at,dia,estaci,nivell_absolut,percentatge_volum_embassat,volum_embassat
0,row-kefq~575y.4dty,rv-rdjg.vtvg-8jwv,2026-10-22T10:00:11.267Z,2026-11-05T11:00:09.027Z,2026-10-22T00:00:00.000,Embassament de la Llosa del Cavall (Navès),801.42,83.9,60.01
1,row-jvef-nqxr_3ujb,rv-uy3d_6v7c~c75v,2026-10-22T10:00:11.267Z,2026-11-05T11:00:09.027Z,2026-10-22T00:00:00.000,Embassament de Siurana (Cornudella de Montsant),480.61,71.5,8.59
2,row-mepu~3j9s~yxue,rv-9khj~qiyx_5jcx,2026-10-23T10:00:11.489Z,2026-11-05T11:00:09.027Z,2026-10-23T00:00:00.000,Embassament de la Llosa del Cavall (Navès),801.44,83.9,60.06
3,row-qqqa.yne6~ka3e,rv-rxnf~xvxy-apzp,2026-10-26T11:00:11.605Z,2026-11-05T11:00:09.027Z,2026-10-26T00:00:00.000,Embassament de la Llosa del Cavall (Navès),801.41,83.8,59.99
4,row-ssbv~i4vi_q9rd,rv-iw2k_2m4s-za28,2026-10-26T11:00:11.605Z,2026-11-05T11:00:09.027Z,2026-10-26T00:00:00.000,Embassament de Siurana (Cornudella de Montsant),480.58,71.4,8.57


## DESCARGAR DATOS DE SEQUÍA

In [4]:
# ACA - Official drought status

url_sequera = "https://analisi.transparenciacatalunya.cat/api/v3/views/i5n8-43cw/query.json"
df_sequera = download_socrata(url_sequera)

print("Dimensions:", df_sequera.shape)

display(df_sequera.head())

Downloaded 8677 records
Dimensions: (8677, 13)


,:id,:version,:created_at,:updated_at,data_canvi_estat_sequera,codi_unitat_explotaci,unitat_explotaci,codi_estat_sequera_hidrol,estat_sequera_hidrol_gic,codi_estat_sequera_pluviom,estat_sequera_pluviom_tric,codi_municipi,municipi
0,row-siqu~mz5e~hh2f,rv-e84a-ywg9~7p97,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170010,Agullana
1,row-t4ph-fgsd-kbsv,rv-eiav_8cu9.i54v,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170118,"Armentera, l'"
2,row-6szz.h62x~xu7q,rv-fjk4-h3ir-rtdk,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170427,Capmany
3,row-zsrx-69ui-8wzf,rv-bsk2.cien-4j8h,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170643,Espolla
4,row-sj7b-amak-5hxn,rv-xznj~5y2f~xf53,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170773,Garriguella


## EXPLORACIÓN DE LOS DATASETS

### DIMENSIONES Y COLUMNAS

In [5]:
datasets = {
    "XEMA": df_xema,
    "ACA reservoirs": df_aca,
    "ACA drought": df_sequera
}

for name, df in datasets.items():

    print("\n" + "="*60)
    print(name)
    print("="*60)

    print("Dimensions:", df.shape)
    print("Columns:", df.columns.tolist())


XEMA
Dimensions: (33475032, 10)
Columns: ['ID', 'CODI_ESTACIO', 'NOM_ESTACIO', 'DATA_LECTURA', 'CODI_VARIABLE', 'NOM_VARIABLE', 'VALOR', 'UNITAT', 'HORA _TU', 'Estat']

ACA reservoirs
Dimensions: (88354, 9)
Columns: [':id', ':version', ':created_at', ':updated_at', 'dia', 'estaci', 'nivell_absolut', 'percentatge_volum_embassat', 'volum_embassat']

ACA drought
Dimensions: (8677, 13)
Columns: [':id', ':version', ':created_at', ':updated_at', 'data_canvi_estat_sequera', 'codi_unitat_explotaci', 'unitat_explotaci', 'codi_estat_sequera_hidrol', 'estat_sequera_hidrol_gic', 'codi_estat_sequera_pluviom', 'estat_sequera_pluviom_tric', 'codi_municipi', 'municipi']


### TIPOS DE VARIABLES

In [6]:
for name, df in datasets.items():

    print("\n" + "="*60)
    print(name)
    print("="*60)

    display(pd.DataFrame({
        "Column": df.columns,
        "Data type": df.dtypes.astype(str).values
    }))


XEMA


,Column,Data type
0,ID,str
1,CODI_ESTACIO,str
2,NOM_ESTACIO,str
3,DATA_LECTURA,str
4,CODI_VARIABLE,float64
5,NOM_VARIABLE,str
6,VALOR,str
7,UNITAT,str
8,HORA _TU,str
9,Estat,str



ACA reservoirs


,Column,Data type
0,:id,str
1,:version,str
2,:created_at,str
3,:updated_at,str
4,dia,str
5,estaci,str
6,nivell_absolut,str
7,percentatge_volum_embassat,str
8,volum_embassat,str



ACA drought


,Column,Data type
0,:id,str
1,:version,str
2,:created_at,str
3,:updated_at,str
4,data_canvi_estat_sequera,str
5,codi_unitat_explotaci,str
6,unitat_explotaci,str
7,codi_estat_sequera_hidrol,str
8,estat_sequera_hidrol_gic,str
9,codi_estat_sequera_pluviom,str


### PRIMERAS FILAS

In [7]:
for name, df in datasets.items():

    print("\n" + "="*60)
    print(name)
    print("="*60)

    display(df.head())


XEMA


,ID,CODI_ESTACIO,NOM_ESTACIO,DATA_LECTURA,CODI_VARIABLE,NOM_VARIABLE,VALOR,UNITAT,HORA _TU,Estat
0,VK1000098801,VK,Raimat,01/09/1988,1.00,Temperatura mitjana diària,"23,3",°C,NaN,Representatiu
1,VK1002098801,VK,Raimat,01/09/1988,1.00,Temperatura mínima diària + hora,"18,3",°C,0:00,Representatiu
2,VK1100098801,VK,Raimat,01/09/1988,1.10,Humitat relativa mitjana diària,58,%,NaN,Representatiu
3,VK1300098801,VK,Raimat,01/09/1988,1.30,Precipitació acumulada diària,"0,6",mm,NaN,Representatiu
4,VK1301098801,VK,Raimat,01/09/1988,1.30,Precipitació acumulada diària (8-8 h),"0,6",mm,NaN,Representatiu



ACA reservoirs


,:id,:version,:created_at,:updated_at,dia,estaci,nivell_absolut,percentatge_volum_embassat,volum_embassat
0,row-kefq~575y.4dty,rv-rdjg.vtvg-8jwv,2026-10-22T10:00:11.267Z,2026-11-05T11:00:09.027Z,2026-10-22T00:00:00.000,Embassament de la Llosa del Cavall (Navès),801.42,83.9,60.01
1,row-jvef-nqxr_3ujb,rv-uy3d_6v7c~c75v,2026-10-22T10:00:11.267Z,2026-11-05T11:00:09.027Z,2026-10-22T00:00:00.000,Embassament de Siurana (Cornudella de Montsant),480.61,71.5,8.59
2,row-mepu~3j9s~yxue,rv-9khj~qiyx_5jcx,2026-10-23T10:00:11.489Z,2026-11-05T11:00:09.027Z,2026-10-23T00:00:00.000,Embassament de la Llosa del Cavall (Navès),801.44,83.9,60.06
3,row-qqqa.yne6~ka3e,rv-rxnf~xvxy-apzp,2026-10-26T11:00:11.605Z,2026-11-05T11:00:09.027Z,2026-10-26T00:00:00.000,Embassament de la Llosa del Cavall (Navès),801.41,83.8,59.99
4,row-ssbv~i4vi_q9rd,rv-iw2k_2m4s-za28,2026-10-26T11:00:11.605Z,2026-11-05T11:00:09.027Z,2026-10-26T00:00:00.000,Embassament de Siurana (Cornudella de Montsant),480.58,71.4,8.57



ACA drought


,:id,:version,:created_at,:updated_at,data_canvi_estat_sequera,codi_unitat_explotaci,unitat_explotaci,codi_estat_sequera_hidrol,estat_sequera_hidrol_gic,codi_estat_sequera_pluviom,estat_sequera_pluviom_tric,codi_municipi,municipi
0,row-siqu~mz5e~hh2f,rv-e84a-ywg9~7p97,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170010,Agullana
1,row-t4ph-fgsd-kbsv,rv-eiav_8cu9.i54v,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170118,"Armentera, l'"
2,row-6szz.h62x~xu7q,rv-fjk4-h3ir-rtdk,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170427,Capmany
3,row-zsrx-69ui-8wzf,rv-bsk2.cien-4j8h,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170643,Espolla
4,row-sj7b-amak-5hxn,rv-xznj~5y2f~xf53,2025-05-17T08:54:07.369Z,2025-05-17T08:54:07.369Z,2025-05-16T00:00:00.000,03,Aqüífer Fluvià - Muga,00004,NORMALITAT,00001,NORMALITAT,170773,Garriguella


### VALORES NULOS

In [8]:
for name, df in datasets.items():

    print("\n" + "="*60)
    print(name)
    print("="*60)

    missing = pd.DataFrame({
        "Missing values": df.isnull().sum(),
        "Percentage (%)": df.isnull().mean() * 100
    })

    display(missing)


XEMA


,Missing values,Percentage (%)
ID,0,0.00
CODI_ESTACIO,0,0.00
NOM_ESTACIO,0,0.00
DATA_LECTURA,0,0.00
CODI_VARIABLE,0,0.00
NOM_VARIABLE,0,0.00
VALOR,0,0.00
UNITAT,0,0.00
HORA _TU,17902103,53.48
Estat,10699,0.03



ACA reservoirs


,Missing values,Percentage (%)
:id,0,0.00
:version,0,0.00
:created_at,0,0.00
:updated_at,0,0.00
dia,0,0.00
estaci,0,0.00
nivell_absolut,53,0.06
percentatge_volum_embassat,0,0.00
volum_embassat,0,0.00



ACA drought


,Missing values,Percentage (%)
:id,0,0.00
:version,0,0.00
:created_at,0,0.00
:updated_at,0,0.00
data_canvi_estat_sequera,0,0.00
codi_unitat_explotaci,0,0.00
unitat_explotaci,0,0.00
codi_estat_sequera_hidrol,0,0.00
estat_sequera_hidrol_gic,0,0.00
codi_estat_sequera_pluviom,0,0.00


### DUPLICADOS

In [9]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicated rows")

XEMA: 0 duplicated rows
ACA reservoirs: 0 duplicated rows
ACA drought: 0 duplicated rows
